# Discovering Music Listening Patterns Using Frequent Itemset Mining
### An Association-Rule-Based Artist Recommender on the Last.fm (HetRec 2011) Dataset

> **📌 OUTLINE / GUIDE NOTEBOOK.** Each **GUIDE** cell lists what the section must contain. It follows the BDCC Lab 1 instructions, your approved game plan, and the class FIM materials (*Frequent Itemset Mining* notebook and the Sessions 3–4 slides). Replace the guide bullets with your write-up, fill in the `TODO` cells, and delete these notes before submitting.

**GUIDE: Title block**
- Keep the game-plan title (above) and a one-line subtitle naming the method: FIM → association rules → recommender.
- ⚠️ **No names, student numbers, or anything that identifies you** (lab rule). Use your SLT label only if allowed.

## Submission Checklist (delete before submitting)
- [ ] The notebook **regenerates the whole report when run top to bottom** (*Kernel → Restart & Run All*).
- [ ] Submit the `.ipynb` **and its HTML export**.
- [ ] Supplementary notebooks / `.py` files (e.g., `fim_utils.py` with your own Apriori/FP-Growth/`assoc` functions) are **referred to in this main notebook** and included.
- [ ] All relevant files, **including images** (e.g., pipeline diagram), are included.
- [ ] **No raw, preprocessed, or processed data files** are included unless a visualization needs one. Read data straight from the shared folder on Jojie.
- [ ] **File paths do not include your username.**
- [ ] No names or identifying info anywhere (cells, file names, plot titles, notebook metadata).
- [ ] The recommender has a **performance evaluation** (lab requirement).
- [ ] Readable for **peer review**: every table and chart has a caption and an interpretation.

## Abstract

**GUIDE**
- 150–250 words covering problem → data → method → key findings → implication. **Write it last.**
- Data: Last.fm HetRec 2011, 1,892 users (transactions), 17,632 artists (items), ~92.8K user–artist records.
- Method: users as transactions and artists as items; Apriori and FP-Growth (pyFIM) at a chosen *minsup*; association rules filtered by *minconf* and ranked by lift; rule-based recommender.
- 1–2 headline numbers: e.g., # frequent itemsets at your minsup, one standout rule with its confidence and lift, recommender hit rate / precision@k vs. the popularity baseline.

## Table of Contents
1. Introduction
2. Problem Statement
3. Motivation
4. Data Source and Description
5. Methodology
6. Data Loading
7. Exploratory Data Analysis
8. Preprocessing: Building the Transaction Database
9. Frequent Itemset Mining
10. Association Pattern Mining
11. Recommender System from Association Rules
12. Performance Evaluation
13. Results and Discussion
14. Conclusion
15. Recommendations
16. References

## 1. Introduction

**GUIDE**
- Context: streaming platforms log what people listen to, and listeners tend to like **groups of related artists**, not artists in isolation. Introduce what Last.FM is
- Introduce FIM in the class framing: FIM and Association Pattern Mining **came from market basket analysis**. Here, *a user's listening history is the basket and the artists are the items*.
- Mention the class point that FIM usually **doesn't end at frequent itemsets**: it continues to association pattern mining, which here feeds a recommender.
- End with a short roadmap of the report.

Streaming platforms compile listening histories through tracking their users' listening activities in large volumes. Listening histories show not only the user’s preference towards particular artists, but also their preference towards certain clusters of artists who have similar genres, eras, voice qualities, and cultures. Last.fm is an online music platform that exhibits such behavior by way of “scrobbling” which involves recording the artists and songs played by the user to generate a profile of listening behavior. (Last.fm, n.d.). The HetRec 2011 Last.fm dataset used in this notebook includes listening records of 1,892 artists, forming a good foundation for analyzing co-listening relationships between 17,632 artists (Cantador et al., 2011; GroupLens, 2013).

Frequent itemset mining and association pattern mining were initially developed for market basket analysis, wherein companies tried to identify items commonly purchased by their customers (Agrawal & Srikant, 1994). In that scenario, each customer’s purchase basket becomes the transaction, and each product becomes the item. This report follows the same approach in the case of music listening, where the listening history of each individual becomes the basket, and the artists listened to become the items. Thus, FIM discovers frequent sets of artists appearing together in users’ listening histories, and association pattern mining derives the rules of the form “If a user listens to artist A, he/she listens to artist B as well” (Agrawal & Srikant, 1994; Aggarwal, 2015).

Under the class setting, FIM is not limited to the identification of frequent itemsets. Frequent itemsets indicate the combination of artists that are frequently found; however, association pattern mining transforms frequent itemsets into meaningful recommendation rules using confidence and lift measures (Agrawal & Srikant, 1994; Fournier-Viger et al., 2017). For instance, an association between Depeche Mode and Martin L. Gore can form the basis for a "fans also like" recommendation. The rule-based technique proves especially valuable for discovering new music since it only uses the listening activity of users, without any need for ratings, and generates recommendations that can be justified with reference to the artists that triggered them. Previous research has also applied association-rule mining techniques like the Apriori algorithm to construct recommendation systems from user-product interaction information (Talwar et al., n.d.).

In this report, we analyze artists who are associated often in the listening history of Last.fm users and assess if such associations can recommend artists that users may enjoy as well. Firstly, the report starts with an introduction to the Last.fm dataset and formation of the user-artist transaction database (Cantador et al., 2011; GroupLens, 2013). The next step includes mining of frequent artist itemsets using Apriori and FP-Growth algorithms (Agrawal & Srikant, 1994; Han et al., 2000). Generation of association rules on the basis of support, confidence, and lift measures followed by generation of artist recommendation from high-quality rules. Performance of the recommender is tested against a baseline method and results are analyzed.


## 2. Problem Statement

**GUIDE**
- Use the game-plan problem: *Which artists frequently occur together in users' listening histories, and can these relationships be used to discover artists a user may also be interested in?*
- Optional sub-questions:
  1. Which artist combinations are frequent (support ≥ *minsup*)?
  2. Which rules $A \rightarrow B$ are reliable (high confidence) **and** genuinely associated (lift > 1)?
  3. How well does a rule-based recommender predict artists a user actually listens to, compared with recommending popular artists?

## 3. Motivation

**GUIDE**
- Who benefits: platforms (discovery, "fans also like", playlists), artists and labels (fan overlap, collaborations, co-marketing, tour line-ups), listeners (finding new music).
- Why FIM and association rules: **interpretable** ("recommended because you listen to X and Y"), need no ratings, only listening presence. This matches the class example of using FIM for recommender systems.

## 4. Data Source and Description

**GUIDE**
- **Source:** HetRec 2011 Last.fm dataset (`hetrec2011-lastfm-2k`), GroupLens: https://grouplens.org/datasets/hetrec-2011/. On Jojie: `hetrec2011/lastfm` (~16 MB).
- **Scope:** 1,892 users · 17,632 artists · 92,834 user–artist records · 11,946 tags · 186,479 tag assignments · 12,717 bi-directional friend relations.
- **File table** (state which files you use and why):

| File | Columns | Description | Used? |
|---|---|---|---|
| `user_artists.dat` | `userID`, `artistID`, `weight` | Artists each user listened to; `weight` = play count | ✅ **Main file**: builds the transaction database |
| `artists.dat` | `id`, `name`, `url`, `pictureURL` | Artist metadata | ✅ Maps IDs → names so itemsets and rules are readable |
| `tags.dat` | `tagID`, `tagValue` | Tag vocabulary (genres, moods…) | Optional: label or interpret itemsets by genre |
| `user_taggedartists.dat` | `userID`, `artistID`, `tagID`, `day`, `month`, `year` | User-assigned artist tags with date | Optional: genre context |
| `user_taggedartists-timestamps.dat` | `userID`, `artistID`, `tagID`, `timestamp` | Same as above, timestamp in ms | Optional (duplicate of above) |
| `user_friends.dat` | `userID`, `friendID` | Friend pairs (each listed both ways) | Probably unused: say so and why |

- **Format:** tab-separated, header row, one record per line. `artists.dat` / `tags.dat` contain non-ASCII names.
- **Data dictionary** for the columns you use (name, type, meaning).
- **Known limitations** to state up front:
  - Each user holds only their **top ~50 artists** (median = max = 50). "Listened to" really means "is in the user's top 50".
  - Data is from ~2011. The most common artists are Lady Gaga (32% of users), Britney Spears, Rihanna, The Beatles and Katy Perry.
- **License:** non-commercial use; cite Last.fm and HetRec'11.

## 5. Methodology

**GUIDE**: Describe the pipeline (ideally as a **flowchart image** included in the submission), then define the concepts **using the class notation**.

**Pipeline**
1. Load data from the shared Jojie folder
2. EDA
3. Build the transaction database $\mathscr{T}$ (one transaction per user)
4. Frequent itemset mining with Apriori and FP-Growth
5. Association pattern mining (rules filtered by *minconf*, ranked by lift)
6. Rule-based recommender
7. Hold-out evaluation against a popularity baseline
8. Insights and recommendations

**Definitions to include (from the class notebook):**
- Database $\mathscr{T} = \{T_1, \dots, T_N\}$; each $T_i$ is a set of items from the universe $\mathscr{U}$, $d = |\mathscr{U}|$. Here $N$ = users and $\mathscr{U}$ = artists. Only **presence** counts and play counts are ignored, as in the class notebook.
- Itemset $X \subseteq \mathscr{U}$; a $k$-itemset has $|X| = k$.
- **Absolute support** (the course's definition of support): $sup(X) = |\{T_i \mid X \subseteq T_i\}|$
- **Relative support:** $relSup(X) = sup(X)/N$. ⚠️ Say which one you report; libraries often mean *relative* support.
- **Frequent itemset:** $sup(X) \geq minsup$.
- **Maximal frequent itemset:** frequent, and no superset is frequent. **Closed itemset:** no superset has the same support.
- **Confidence:** $conf(A \rightarrow B) = \dfrac{sup(A \cup B)}{sup(A)}$, with antecedent $A$ and consequent $B$.
- **Lift:** $lift(A \rightarrow B) = \dfrac{conf(A \rightarrow B)}{relSup(B)}$. Above 1 = positive association, 1 = none, below 1 = negative.

**Algorithms (explain briefly in your own words):**
- **Brute force:** evaluates all $2^d - 1$ itemsets. With $d = 17{,}632$ artists this is infeasible, which is why it is excluded (cite the class runtime argument: exponential in items, linear in transactions).
- **Apriori:** breadth-first search over the itemset lattice using **support monotonicity** and the **downward closure property** to prune candidates.
- **FP-Growth:** pattern growth on a prefix tree using **projected databases**, so it never generates candidates that are absent from the data.
- **Tools:** `pandas`, `numpy`, `scipy.sparse.csr_matrix`, `matplotlib`, and **pyFIM** (`import fim`), the library used in class. Optionally reuse your own `apriori`, `fpgrowth`, and `assoc` functions from the class exercises in a supplementary `fim_utils.py`, and say so here.

## 6. Data Loading

**GUIDE**
- One imports cell at the top. Install pyFIM on Jojie if needed (`pip install pyfim`).
- Read **directly from the shared Jojie folder**. The path must **not** contain your username. Confirm the exact public path on Jojie.
- `pd.read_csv(path, sep='\t')`; pass `encoding=` if names fail to load.
- Show `.shape`, `.head()`, `.info()` per table with a one-line description.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import fim  # pyFIM (used in class)

# Optional supplementary module with your class-exercise implementations
# from fim_utils import apriori as my_apriori, fpgrowth as my_fpgrowth, assoc

In [ ]:
DATA_DIR = '/mnt/data/public/hetrec2011/lastfm/'  # TODO: confirm actual shared path (no username!)

# user_artists = pd.read_csv(DATA_DIR + 'user_artists.dat', sep='\t')
# artists      = pd.read_csv(DATA_DIR + 'artists.dat', sep='\t')
# tags         = pd.read_csv(DATA_DIR + 'tags.dat', sep='\t', encoding='latin-1')   # optional
# user_tagged  = pd.read_csv(DATA_DIR + 'user_taggedartists.dat', sep='\t')         # optional

## 7. Exploratory Data Analysis

**GUIDE**: Every chart needs a title, axis labels, and a caption with interpretation.
- **Quality checks:** missing values, duplicate (user, artist) pairs, artist IDs missing from `artists.dat`, duplicate artist names.
- **Basic sizes:** $N$ (users), $d$ (artists), records, and **density** of the $N \times d$ binary matrix (very sparse, which is why a `csr_matrix` representation fits).
- **Transaction length** $n_i = |T_i|$: histogram showing that almost every user has exactly 50. Explain what that means.
- **Item support distribution** (users per artist, log scale): long tail, with most artists having support 1–2.
- **Top-N artists by support** (bar chart, both absolute and relative support).
- **Artists remaining at each candidate minsup** (e.g., 1%, 2%, 3%, 5%, 10%). For reference: 838 artists at 1%, 432 at 2%, 135 at 5%, 47 at 10%. This feeds directly into the minsup choice.
- *(Optional)* `weight` distribution (heavily skewed, use a log scale) and top tags/genres.
- Close with **"Key EDA takeaways"** that motivate the preprocessing.

In [ ]:
# TODO: quality checks + basic sizes + density

In [ ]:
# TODO: transaction-length histogram; item-support distribution (log scale)

In [ ]:
# TODO: top-N artists bar chart; #artists surviving each minsup

## 8. Preprocessing: Building the Transaction Database

**GUIDE**
- Represent $\mathscr{T}$ **as in the class notebook**: a **list of sets**, one set of artist names per user.
- Optionally also build the **sparse binary matrix** (`csr_matrix`, columns sorted alphabetically) like `to_sparse` from class, to report shape and density.
- Map `artistID` → `name` (artist names are unique in `artists.dat`, but verify this).
- Justify any filtering, e.g., dropping artists with support below *minsup* (they can never be frequent), or *(optional)* counting an artist only if `weight` passes a threshold.
- **Train/test split, needed for Section 12. Decide it here:**
  - Split users into train and test (e.g., 80/20, with `random_state` fixed for reproducibility).
  - For each test user, **hide** a portion of their artists (e.g., 20%) as ground truth and keep the rest as input.
  - **Mine itemsets and rules on the training transactions only** to avoid leakage.
- Show a few sample transactions and the final $N$, $d$.

In [ ]:
# TODO: db = list of sets of artist names (one per user)

In [ ]:
# TODO: optional csr_matrix representation (shape, density)

In [ ]:
# TODO: train/test split; hide part of each test user's artists

## 9. Frequent Itemset Mining

**GUIDE**
- Run **Apriori and FP-Growth** (`fim.apriori`, `fim.fpgrowth`) on the training database.
  - pyFIM note from class: **`supp` > 0 is a percentage; `supp` < 0 is absolute support**. For example, `supp=-57` means at least 57 users (~3%).
  - `zmin=2` keeps itemsets of 2+ artists; `zmax` caps the itemset size.
- **Choosing minsup:** try several values and tabulate **# frequent itemsets and runtime for each algorithm**. Reference counts on the full data: 5% → 8,435 itemsets, 3% → 66,579, 2% → 324,088. The count explodes quickly because pop artists co-occur densely. Pick a value that is meaningful and manageable, and justify it.
- **Apriori vs FP-Growth:**
  - Confirm both return **the same itemsets**.
  - Compare runtime with a table or bar chart, and link it to the class discussion (FP-Growth avoids candidate generation, which matters when there are many items).
- **Validate your class implementation (optional but strong):** run your own `apriori`/`fpgrowth` from the class exercises on a **small subset** (e.g., top-30 artists, or a high minsup) and assert they match pyFIM. They are pure-Python and will be too slow on the full data, so say this.
- Report:
  - # frequent itemsets by size $k$
  - # **maximal** (`target='m'`) and **closed** (`target='c'`) itemsets, and why they compactly summarize the result
  - Top frequent $k$-itemsets ($k \geq 2$) with absolute and relative support
- **Interpret:** do the itemsets form recognizable taste clusters (pop divas, 80s synth-pop, metal…)? Tags can help label them.

In [ ]:
# TODO: grid over minsup -> counts & runtime for fim.apriori vs fim.fpgrowth
# e.g. t=time.time(); fi = fim.fpgrowth(db_train, supp=-57, zmin=1); time.time()-t

In [ ]:
# TODO: final itemsets at chosen minsup; counts by size; maximal (target='m') & closed (target='c')

In [ ]:
# TODO (optional): check your own class functions match pyFIM on a small subset

## 10. Association Pattern Mining

**GUIDE**
- Generate rules with pyFIM, e.g. `fim.fpgrowth(db_train, target='r', supp=..., conf=..., zmin=2, report='acl')`
  - `report` codes: `a` = absolute support, `s` = relative support, `c` = confidence, `l` = lift.
  - ⚠️ pyFIM returns rules as **`(consequent, antecedent, ...)`**, consequent first. Convert to a DataFrame with explicit `antecedent` / `consequent` columns, like the class `assoc` output.
  - Or use your own `assoc(fi, minconf, db_size)` from class on the frequent itemsets, and cross-check it with pyFIM.
- **Choose and justify `minconf`.** Expect huge rule counts: on the full data, minsup 3% with minconf 50% gives over 1 million rules. Control this with a higher minsup, `zmax`, a lift threshold, or by keeping only single-item consequents.
- Present **top rules by lift and by confidence**, sorted like the class `assoc` (lift → confidence → support). Explain a few in plain English, e.g. *"users who listen to Dave Gahan and Depeche Mode also tend to listen to Martin L. Gore (conf 0.62, lift ≈ 28)"*.
- Discuss **popularity bias:** rules ending in mainstream artists (e.g., Lady Gaga) can have high confidence but lift near 1, so explain why lift matters for recommendations.
- Visualize: support vs. confidence scatter colored by lift; *(optional)* a network graph of top rules, or a lift heatmap of top artist pairs.

In [ ]:
# TODO: rules = fim.fpgrowth(db_train, target='r', supp=..., conf=..., zmin=2, report='acl')

In [ ]:
# TODO: to DataFrame (antecedent, consequent, support, confidence, lift); sort; filter lift > 1

In [ ]:
# TODO: plots + top-rule tables

## 11. Recommender System from Association Rules

**GUIDE**
- Explain the logic step by step:
  1. Input: the set of artists a user listens to.
  2. Select rules whose **antecedent ⊆ user's artists**.
  3. Keep consequents the user **doesn't already have**.
  4. **Score and rank** them (e.g., max confidence, max lift, or the sum of confidences over matching rules). Justify the choice.
  5. Return the top-*k*.
- Write a reusable function (it can go in a supplementary `.py` file; reference it here).
- Edge case: users with no matching rule fall back to the **most popular unheard artists**. Report how often this happens.
- **Demo:** 2–3 anonymous test users showing their input artists, the recommendations, and the rule that triggered each one (explainability is a key selling point).

In [ ]:
def recommend(user_items, rules_df, k=10):
    """TODO: top-k artists from rules whose antecedent is a subset of user_items."""
    pass

In [ ]:
# TODO: demo for a few test users

## 12. Performance Evaluation

**GUIDE**: ⚠️ **Required by the lab** ("Recommender systems should have a performance evaluation").
- Restate the protocol from Section 8: rules come from training users; for each test user, the visible artists are the input and the hidden artists are the ground truth.
- Metrics:
  - **Precision@k**, **Recall@k**, **Hit rate@k** (≥ 1 hidden artist recommended), optionally F1@k
  - **Coverage:** % of test users getting at least one rule-based recommendation; % of artists ever recommended
- **Baseline:** a popularity recommender (top-k most popular artists the user hasn't heard). The rule-based recommender should beat it, so state whether it does.
- *(Optional)* Sensitivity of the metrics to minsup, minconf, and k (line plots).
- Show a results table and chart, then interpret: where does it work well (mainstream tastes) and where does it fail (niche users or artists below minsup)?

In [ ]:
# TODO: evaluation loop -> precision@k, recall@k, hit rate@k, coverage

In [ ]:
# TODO: popularity baseline + comparison table/plot

In [ ]:
# TODO (optional): sensitivity analysis

## 13. Results and Discussion

**GUIDE**
- Synthesize what the results **mean**; don't just repeat tables.
- Answer each sub-question from the Problem Statement explicitly.
- Discuss the algorithm comparison (Apriori vs FP-Growth runtime on this data) and the effect of the minsup/minconf choices.
- **Limitations:**
  - Top-50 cap per user, and presence only (play counts ignored)
  - Popularity bias, so the long tail of niche artists never reaches minsup
  - Small, 2011-era user base, so findings may not generalize
  - Results are sensitive to thresholds

## 14. Conclusion

**GUIDE**
- 1–2 paragraphs: problem, approach, key findings with numbers, and a clear verdict on whether association rules are a viable basis for artist recommendations on this data.

## 15. Recommendations

**GUIDE**: Give actionable suggestions, like Problem 10 in the class notebook (three concrete suggestions backed by specific itemsets or rules with their numbers).
- **For the platform / stakeholders**, e.g.:
  - Use high-lift rules for "Fans also like" or auto-playlists
  - Bundle or co-promote artists with strong associations
  - Use the rules to seed new users (cold start) from one or two chosen artists
- **Future work**, e.g.:
  - Use `weight` (count an artist only above a play threshold, or weight the rules)
  - Compare against collaborative filtering (item-based or matrix factorization)
  - Mine **tag/genre** itemsets to cover niche artists
  - Use `user_friends.dat` for social recommendations

## 16. References

**GUIDE**: Use one consistent citation style.
- Cantador, I., Brusilovsky, P., & Kuflik, T. (2011). 2nd Workshop on Information Heterogeneity and Fusion in Recommender Systems (HetRec 2011). *Proc. 5th ACM Conference on Recommender Systems (RecSys 2011)*. ACM.
- Last.fm, http://www.lastfm.com · GroupLens HetRec 2011, https://grouplens.org/datasets/hetrec-2011/
- Aggarwal, C. (2015). *Data Mining: The Textbook*. Springer.
- Fournier-Viger, P., Lin, J. C. W., Vo, B., Chi, T. T., Zhang, J., & Le, H. B. (2017). A survey of itemset mining. *WIREs Data Mining and Knowledge Discovery, 7*(4), e1207.
- Borgelt, C. *PyFIM: Frequent Item Set Mining for Python*. http://www.borgelt.net/pyfim.html
- Agrawal, R., & Srikant, R. (1994). Fast algorithms for mining association rules. *Proc. VLDB.* (Apriori)
- Han, J., Pei, J., & Yin, Y. (2000). Mining frequent patterns without candidate generation. *Proc. ACM SIGMOD.* (FP-Growth)
- Course materials: *Frequent Itemset Mining* notebook (C. Alis) and BDCC Sessions 3–4 slides.
- GroupLens. (2013). HetRec 2011: Last.fm dataset. https://grouplens.org/datasets/hetrec-2011/
- Talwar, K. S., Oraganti, A., Mahajan, N., & Narsale, P. (n.d.). Recommendation system using Apriori algorithm.